# Supply Chain Late Delivery Prediction - Feature Engineering & Feature Selection
**Student:** IT24104225 - Weerathunga W.M.C.N.  
**Assigned Responsibility:** Feature Engineering & Feature Selection  
**Target Variable:** `Late_delivery_risk` (1 = Late delivery, 0 = On-time / Advance)  

---

## 1. Project Setup & Initial Dataset Inspection
I first loaded the raw DataCo Supply Chain dataset and checked its structure, column types, missing values, and the target distribution before designing any feature engineering or selection steps.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

# Set display options for clean inspection
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 1000)

DATA_PATH = "data/DataCoSupplyChainDataset.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset loaded successfully.")
print("Dataset Shape:", df.shape)


Dataset loaded successfully.
Dataset Shape: (180519, 53)



In [2]:
# Inspect target variable, missing values, and duplicate rows
TARGET = "Late_delivery_risk"

print("Target column:", TARGET)
print("\nTarget distribution:")
print(df[TARGET].value_counts())

print("\nTarget proportions:")
print(df[TARGET].value_counts(normalize=True))

print("\nMissing values summary (columns with missing values):")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\nDuplicate full rows count:", df.duplicated().sum())
print("Total unique Order Ids:", df["Order Id"].nunique())


Target column: Late_delivery_risk

Target distribution:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Target proportions:
Late_delivery_risk
1    0.548291
0    0.451709
Name: proportion, dtype: float64

Missing values summary (columns with missing values):
Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Duplicate full rows count: 0
Total unique Order Ids: 65752



## 2. Grouped Train / Test Split
I defined `Late_delivery_risk` as the target variable. Because an individual customer order can contain multiple item records, rows sharing the same `Order Id` are correlated. To prevent data leakage across splits, I used `Order Id` as the grouping variable with an 80/20 grouped split using `GroupShuffleSplit`.

I then verified that no `Order Id` appears in both the training set and the testing set.


In [3]:
# Perform grouped train/test split (80% train, 20% test) grouped by Order Id
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["Order Id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Verify Order Id grouping and check for any overlap
train_orders = set(train_df["Order Id"])
test_orders = set(test_df["Order Id"])
overlap = train_orders.intersection(test_orders)

print(f"Unique Order Ids in Train: {len(train_orders):,}")
print(f"Unique Order Ids in Test:  {len(test_orders):,}")
print(f"Overlap in Order Ids between Train and Test: {len(overlap)}")

# Verify target balance across splits
print("\nTrain Target Proportions:")
print(train_df[TARGET].value_counts(normalize=True).round(4))

print("\nTest Target Proportions:")
print(test_df[TARGET].value_counts(normalize=True).round(4))


Train shape: (144650, 53)
Test shape: (35869, 53)
Unique Order Ids in Train: 52,601
Unique Order Ids in Test:  13,151
Overlap in Order Ids between Train and Test: 0

Train Target Proportions:
Late_delivery_risk
1    0.5483
0    0.4517
Name: proportion, dtype: float64

Test Target Proportions:
Late_delivery_risk
1    0.5484
0    0.4516
Name: proportion, dtype: float64

